In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from scipy.sparse import hstack

In [2]:
df = pd.read_csv("../data/processed/english_hindi.csv")

X = df["clean_text"]
y = df["labels"]

print(df.shape)
print(df.head())

(10287, 6)
  labels                                               text     lang  \
0   spam  No 1 POLYPHONIC tone 4 ur mob every week! Just...  english   
1    ham     हाय यह अच्छी तरह से जाने के लिए मेरा शुक्रिया!    hindi   
2    ham  मैं तुम्हें बाद में फोन करता हूँ, नेटवर्क नहीं...    hindi   
3    ham                                  भारत ले जाना है:)    hindi   
4   spam  CURGERAN! आपके मोबाइल नंबर को 5 पाउंड इनाम GUR...    hindi   

  sentiment  message_length                                         clean_text  
0  Positive             147  no 1 polyphonic tone 4 ur mob every week! just...  
1   Neutral              46     हाय यह अच्छी तरह से जाने के लिए मेरा शुक्रिया!  
2   Neutral              50  मैं तुम्हें बाद में फोन करता हूँ, नेटवर्क नहीं...  
3   Neutral              17                                  भारत ले जाना है:)  
4   Neutral             130  curgeran! आपके मोबाइल नंबर को 5 पाउंड इनाम gur...  


In [3]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(label_encoder.classes_)

['ham' 'spam']


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training:", len(X_train))
print("Testing:", len(X_test))

Training: 8229
Testing: 2058


In [6]:
char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2,
    max_df=0.95
)

X_train_char = char_vectorizer.fit_transform(X_train)
X_test_char = char_vectorizer.transform(X_test)

print("Character features:", X_train_char.shape[1])

Character features: 119013


In [7]:
svm_char = LinearSVC()

svm_char.fit(X_train_char, y_train)

y_pred_char = svm_char.predict(X_test_char)

In [8]:
char_accuracy = accuracy_score(y_test, y_pred_char)
char_precision = precision_score(y_test, y_pred_char)
char_recall = recall_score(y_test, y_pred_char)
char_f1 = f1_score(y_test, y_pred_char)

print("Character TF-IDF + SVM")
print("Accuracy :", char_accuracy)
print("Precision:", char_precision)
print("Recall   :", char_recall)
print("F1 Score :", char_f1)

Character TF-IDF + SVM
Accuracy : 0.9849368318756074
Precision: 0.9955555555555555
Recall   : 0.8818897637795275
F1 Score : 0.9352818371607515


In [9]:
word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

X_train_word = word_vectorizer.fit_transform(X_train)
X_test_word = word_vectorizer.transform(X_test)

print("Word features:", X_train_word.shape[1])

Word features: 13927


In [10]:
X_train_combined = hstack([
    X_train_word,
    X_train_char
])

X_test_combined = hstack([
    X_test_word,
    X_test_char
])

print("Combined features:", X_train_combined.shape[1])

Combined features: 132940


In [11]:
svm_combined = LinearSVC()

svm_combined.fit(X_train_combined, y_train)

y_pred_combined = svm_combined.predict(X_test_combined)

In [12]:
combined_accuracy = accuracy_score(y_test, y_pred_combined)
combined_precision = precision_score(y_test, y_pred_combined)
combined_recall = recall_score(y_test, y_pred_combined)
combined_f1 = f1_score(y_test, y_pred_combined)

print("Word + Character TF-IDF + SVM")
print("Accuracy :", combined_accuracy)
print("Precision:", combined_precision)
print("Recall   :", combined_recall)
print("F1 Score :", combined_f1)

Word + Character TF-IDF + SVM
Accuracy : 0.9815354713313897
Precision: 0.9864864864864865
Recall   : 0.8622047244094488
F1 Score : 0.9201680672268907


In [13]:
results = pd.DataFrame({
    "Experiment": [
        "Word TF-IDF",
        "Character TF-IDF",
        "Word + Character TF-IDF"
    ],
    "Model": [
        "Linear SVM",
        "Linear SVM",
        "Linear SVM"
    ],
    "Accuracy": [
        0.958212,
        char_accuracy,
        combined_accuracy
    ],
    "Precision": [
        0.838710,
        char_precision,
        combined_precision
    ],
    "Recall": [
        0.818898,
        char_recall,
        combined_recall
    ],
    "F1 Score": [
        0.828685,
        char_f1,
        combined_f1
    ]
})

results

,Experiment,Model,Accuracy,Precision,Recall,F1 Score
0,Word TF-IDF,Linear SVM,0.958212,0.838710,0.818898,0.828685
1,Character TF-IDF,Linear SVM,0.984937,0.995556,0.881890,0.935282
2,Word + Character TF-IDF,Linear SVM,0.981535,0.986486,0.862205,0.920168


In [14]:
results.to_csv("../results_person2.csv", index=False)

print("Results saved successfully!")

Results saved successfully!


In [15]:
print(df.columns)

Index(['labels', 'text', 'lang', 'sentiment', 'message_length', 'clean_text'], dtype='str')


In [16]:
print(df["lang"].value_counts())

lang
english    5155
hindi      5132
Name: count, dtype: int64


In [17]:
english_df = df[df["lang"] == "english"].copy()

print("English messages:", len(english_df))
print(english_df["labels"].value_counts())

English messages: 5155
labels
ham     4514
spam     641
Name: count, dtype: int64


In [18]:
X_eng = english_df["clean_text"]
y_eng = english_df["labels"]

eng_encoder = LabelEncoder()
y_eng_encoded = eng_encoder.fit_transform(y_eng)

print(eng_encoder.classes_)

['ham' 'spam']


In [19]:
X_eng_train, X_eng_test, y_eng_train, y_eng_test = train_test_split(
    X_eng,
    y_eng_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_eng_encoded
)

print("English training:", len(X_eng_train))
print("English testing:", len(X_eng_test))

English training: 4124
English testing: 1031


In [20]:
eng_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

X_eng_train_tfidf = eng_vectorizer.fit_transform(X_eng_train)
X_eng_test_tfidf = eng_vectorizer.transform(X_eng_test)

print("English TF-IDF features:", X_eng_train_tfidf.shape[1])

English TF-IDF features: 9988


In [21]:
svm_eng = LinearSVC()

svm_eng.fit(X_eng_train_tfidf, y_eng_train)

y_eng_pred = svm_eng.predict(X_eng_test_tfidf)

In [22]:
eng_accuracy = accuracy_score(y_eng_test, y_eng_pred)
eng_precision = precision_score(y_eng_test, y_eng_pred)
eng_recall = recall_score(y_eng_test, y_eng_pred)
eng_f1 = f1_score(y_eng_test, y_eng_pred)

print("English-only + Word TF-IDF + SVM")
print("Accuracy :", eng_accuracy)
print("Precision:", eng_precision)
print("Recall   :", eng_recall)
print("F1 Score :", eng_f1)

English-only + Word TF-IDF + SVM
Accuracy : 0.9815712900096993
Precision: 0.957983193277311
Recall   : 0.890625
F1 Score : 0.9230769230769231
